In [1]:
!pip install -q peft groq sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive, userdata
drive.mount("/content/drive")
print("drive mounted")

Mounted at /content/drive
drive mounted


In [3]:
import os
import sys
os.environ["GROQ_API_KEY"]=userdata.get("GROQ_API_KEY")

sys.path.insert(0,"/content/drive/MyDrive/code-debugger/ml_service/model")
sys.path.insert(0,"/content/drive/MyDrive/code-debugger/backend")


**Loads all the modules**

In [4]:
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification,AutoTokenizer,AutoModel
from peft import PeftModel,PeftConfig
from sentence_transformers import SentenceTransformer
from groq_fixer import fix_with_groq,generate_diff

In [5]:
LABELS = ["Syntax Error", "Runtime Error", "Logic Bug", "Multiple Issues"]
SAVE_PATH = "/content/drive/MyDrive/code-debugger/ml_service/model/lora_weights"

# Classifier
config = PeftConfig.from_pretrained(SAVE_PATH)
base = AutoModelForSequenceClassification.from_pretrained(
    config.base_model_name_or_path, num_labels=len(LABELS)
)
classifier = PeftModel.from_pretrained(base, SAVE_PATH)
classifier.eval()
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

# Embedding model
emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Attention model
attn_model = AutoModel.from_pretrained(
    "distilbert-base-uncased", output_attentions=True
)
attn_model.eval()

# In-memory similarity index
index = []

print("All models loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded


**Helper Func**

In [6]:
def classify(code: str) -> dict:
    inputs = tokenizer(code, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = classifier(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().tolist()
    pred_idx = torch.argmax(logits).item()
    return {
        "label": LABELS[pred_idx],
        "confidence": round(probs[pred_idx], 3),
        "all_scores": {LABELS[i]: round(p, 3) for i, p in enumerate(probs)},
    }

def add_to_index(code: str, fix: str, explanation: str):
    emb = emb_model.encode(code, normalize_embeddings=True)
    index.append({"code": code, "fix": fix, "explanation": explanation, "emb": emb})

def search_similar(query_code: str, top_k: int = 3) -> list:
    if not index:
        return []
    q_emb = emb_model.encode(query_code, normalize_embeddings=True)
    scores = [float(np.dot(q_emb, item["emb"])) for item in index]
    ranked = sorted(zip(scores, index), key=lambda x: x[0], reverse=True)
    return [
        {"score": round(s, 3), "code": item["code"], "explanation": item["explanation"]}
        for s, item in ranked[:top_k]
    ]

def get_attention(code: str) -> dict:
    inputs = tokenizer(code, return_tensors="pt", truncation=True, max_length=64)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
    with torch.no_grad():
        outputs = attn_model(**inputs)
    last_layer = outputs.attentions[-1][0]
    avg_attn = last_layer.mean(dim=0).tolist()
    return {
        "tokens": tokens,
        "matrix": avg_attn,
        "num_layers": len(outputs.attentions),
        "num_heads": int(last_layer.shape[0]),
    }

print(" Helper functions ready")

 Helper functions ready


**main pipeline**

In [9]:
def debug(code: str) -> dict:
    print(f"Step 1/4 — Classifying...")
    clf = classify(code)

    print(f"Step 2/4 — Searching similar bugs...")
    similar = search_similar(code)

    print(f"Step 3/4 — Fixing with Groq...")
    fix = fix_with_groq(code, clf["label"])

    print(f"Step 4/4 — Generating diff + attention...")
    diff = generate_diff(code, fix["fixed_code"])
    attention = get_attention(code)

    if fix["fixed_code"]:
        add_to_index(code, fix["fixed_code"], fix["explanation"])

    return {
        "classification": clf,
        "explanation": fix["explanation"],
        "fixed_code": fix["fixed_code"],
        "diff": diff,
        "similar_bugs": similar,
        "attention": attention,
    }

print("Pipeline ready")

Pipeline ready


In [10]:
tests = [
    ("def greet(name)\n    print('Hello')", "Syntax Error"),
    ("x = [1,2,3]\nprint(x[10])", "Runtime Error"),
    ("def is_even(n):\n    return n % 2 == 1", "Logic Bug"),
    ("def foo(n)\n    if n = 0:\n        return 1", "Multiple Issues"),
]

for code, expected in tests:
    print(f"\n{'='*50}")
    print(f"Expected: {expected}")
    result = debug(code)
    print(f"Got:      {result['classification']['label']} ({result['classification']['confidence']*100:.1f}%)")
    print(f"Fix:      {result['fixed_code'][:60]}...")
    print(f"Diff:     {len(result['diff'])} lines")
    print(f"Similar:  {len(result['similar_bugs'])} bugs found")


Expected: Syntax Error
Step 1/4 — Classifying...
Step 2/4 — Searching similar bugs...
Step 3/4 — Fixing with Groq...
Step 4/4 — Generating diff + attention...
Got:      Syntax Error (88.6%)
Fix:      def greet(name):
    print('Hello')...
Diff:     98 lines
Similar:  0 bugs found

Expected: Runtime Error
Step 1/4 — Classifying...
Step 2/4 — Searching similar bugs...
Step 3/4 — Fixing with Groq...
Step 4/4 — Generating diff + attention...
Got:      Runtime Error (99.1%)
Fix:      x = [1,2,3]
print(x[0])  # Accessing the first element
# or
...
Diff:     248 lines
Similar:  1 bugs found

Expected: Logic Bug
Step 1/4 — Classifying...
Step 2/4 — Searching similar bugs...
Step 3/4 — Fixing with Groq...
Step 4/4 — Generating diff + attention...
Got:      Logic Bug (99.8%)
Fix:      def is_even(n):
    return n % 2 == 0...
Diff:     105 lines
Similar:  2 bugs found

Expected: Multiple Issues
Step 1/4 — Classifying...
Step 2/4 — Searching similar bugs...
Step 3/4 — Fixing with Groq...
Step 4/4

In [11]:
import shutil, os

path = "/content/drive/MyDrive/Colab Notebooks/"
for f in os.listdir(path):
    if "pipeline" in f.lower() or "06" in f.lower():
        src = os.path.join(path, f)
        dst = "/content/drive/MyDrive/code-debugger/notebooks/06_pipeline.ipynb"
        shutil.copy(src, dst)
        print(f" Saved: {f}")

 Saved: 06_pipeline.ipynb
